# Week 3 Day 2 Lab 2 (free route) - The Gateway and the Policy

## Deploying UnderCover on Render

On the AWS route, Lab 2 deploys the UnderCover app to Lambda and puts an AgentCore Gateway in front of it, with a Policy that refuses any payment over $1,000. On the free route we deploy the same app as a free web service on Render, with Supabase behind it as it is now.

The Gateway and the Policy are AWS services, so for those you can watch me. Your app on Render still has the guardrails in its own code: the agent's routes need the API key, and a payment over $1,000 is refused whoever asks for it.

This notebook has no code to run. It gives you the steps, with links to Render's own documentation.

## 5 steps:

STEP 1: Choose what to deploy

STEP 2: Create the web service

STEP 3: Add the environment variables

STEP 4: Deploy and sign in

STEP 5: Check the API key

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This is the free version of today's second lab. The main lab, <a href="week3day2_lab2.ipynb">week3day2_lab2.ipynb</a>, runs on AWS. You need the Render account from Day 1, and today's build of the product, yours or <code>week3day2-end</code>.<br/><br/>
            I do this week on AWS, so I'm not recording the Render steps. Render's website changes from time to time, so if a screen looks different from what's described here, follow the linked page in Render's documentation.
            </span>
        </td>
    </tr>
</table>

## Step 1: Choose what to deploy

Render builds the app from a branch of a GitHub repository.

**To deploy my build,** use the course repository, `https://github.com/ed-donner/undercover`, and the branch `week3day2-end`. When you create the web service, choose **Public Git Repository** and paste the address. Render doesn't redeploy a public repository automatically when it changes, which suits a course.

**To deploy your own build,** your branch needs to be in a GitHub repository of your own. Create an empty private repository on GitHub, then push your branch to it from the `undercover` folder:

```
git remote add mine https://github.com/<your-github-name>/undercover.git
git push mine <your-branch>
```

When you create the web service, choose **Git Provider** and pick that repository. Render can see it through the GitHub connection you made on Day 1.

Render's documentation: [Deploy your own code](https://render.com/docs/web-services#deploy-your-own-code).

## Step 2: Create the web service

In the [Render Dashboard](https://dashboard.render.com), choose **New**, then **Web Service**, and pick the repository from Step 1. Then fill in the form:

| Field | Value |
|---|---|
| Name | `undercover` (your service's address is made from this name) |
| Language | Python 3 |
| Branch | `week3day2-end`, or your own branch |
| Region | The one closest to your Supabase project, such as Virginia for a project in us-east-1 |
| Build Command | `uv sync --frozen --only-group server && npm ci --ignore-scripts && npm run build` |
| Start Command | `uv run --no-sync uvicorn backend.main:app --host 0.0.0.0 --port $PORT` |
| Compute | $0 / month |

If the Build Command box already holds a command, replace it with the one above. Under **Compute**, Render selects **$7 / month** to begin with. Choose **$0 / month**, the free instance:

![Render's Compute section with $0 / month selected](../assets/render_compute_free.png)

Under **Advanced**, set the **Health Check Path** to `/api/health`, so Render checks that the app answers before it sends it any traffic.

What the commands do:

- Render adds uv to a Python service when the repository has a `uv.lock` file, and every Render runtime includes Node and npm. So one service installs the backend's packages with `uv sync` and builds the dashboard with `npm run build`, as you do on your laptop.
- `--only-group server` installs only the packages the backend needs on a server, which are listed in the `server` group in `pyproject.toml`. The labs' packages, such as the agent SDKs, stay off the server.
- `--ignore-scripts` skips the repository's setup script, which installs Git hooks and a browser for the tests. A server doesn't need either.
- Render reads the Python version from the repository's `.python-version` file and the Node version from its `.nvmrc` file, and gives the app its port in `$PORT`.

Render's documentation: [Web services](https://render.com/docs/web-services), [Native runtimes](https://render.com/docs/native-runtimes), [Setting your uv version](https://render.com/docs/uv-version), [Setting your Python version](https://render.com/docs/python-version), [Setting your Node.js version](https://render.com/docs/node-version), [Health checks](https://render.com/docs/health-checks), [Regions](https://render.com/docs/regions).

## Step 3: Add the environment variables

In the same form, under **Environment Variables**, add these four lines from your `.env` file, copied as they are. To add them in one go, choose **Add from .env** and paste just these four lines.

| Key | Value |
|---|---|
| `DATABASE_URL` | The same session pooler address as in your `.env` |
| `VITE_SUPABASE_URL` | From your `.env` |
| `VITE_SUPABASE_PUBLISHABLE_KEY` | From your `.env` |
| `UNDERCOVER_API_KEY` | From your `.env` |

If you built the product yourself, your coding agent added `UNDERCOVER_API_KEY` to your `.env`. If you're using my build on the `week3day2-end` branch, add it yourself. Letters and numbers are safest, and this prints a random one in a terminal:

```
uv run python -c "import secrets; print(secrets.token_hex(32))"
```

If your `.env` sets `MODEL_NAME` for free models, add that line too.

Why these:

- The backend needs the database and the API key, and checks the handlers' sign-ins against your Supabase project.
- The dashboard's build reads the two `VITE_SUPABASE` lines. Render's builds can read the environment variables you set, so they go in like the others. If you change either of them later, choose **Save, rebuild, and deploy**, because the build writes them into the dashboard.
- The app is the one place that reads `MODEL_NAME`, and it names the model for everything in the product, so the app on Render needs it to use the same free model as your laptop.
- Keep `DATABASE_URL` on the session pooler, as on your laptop. Render runs one copy of the app that stays up, so it can keep its database connections open.

Render's documentation: [Environment variables and secrets](https://render.com/docs/configure-environment-variables).

## Step 4: Deploy and sign in

Choose **Deploy web service**. The first build takes several minutes, while uv installs the project's packages and npm installs the dashboard's. You can follow it on the service's **Deploys** page, which also has the logs if something goes wrong.

When the deploy is live, open your service's address, which ends in `onrender.com`, and sign in as Dana Whitfield. You'll see the same claims as on your laptop, since both read the same Supabase database.

The agent's service is still on your laptop, where the deployed app can't reach it until Day 4. So for now, dispatches, evaluations and rounds from the deployed dashboard say that the agent's service isn't running. A sign-off works, and pays the claim.

The free service spins down after 15 minutes without a request, so the first page after a break takes about a minute to load.

Render's documentation: [Deploys](https://render.com/docs/deploys), [Free instances](https://render.com/docs/free).

## Step 5: Check the API key

Open these two addresses in your browser, with your service's address in place of `<your-app>`:

- `https://<your-app>.onrender.com/api/health` answers `{"status":"ok"}`.
- `https://<your-app>.onrender.com/api/claims/CLM-1901` is refused with a 401, because your browser doesn't send the agent's API key.

The agent's routes need the key, the handlers sign in, and the backend refuses any payment over $1,000, whoever asks for it. That's the lightweight end of the gateway choices from today's theory: the checks live in the app itself. On the AWS route the Gateway adds a Policy on every tool call in front of these, which you can watch in the main lab.

There's nothing to tear down. A free web service costs nothing, so keep it for Day 3. You can delete it from the Render Dashboard whenever you no longer need it.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Challenge: watch the AWS lab's Part 3, where the Policy refuses a payment over $1,000 before it reaches the app. Then write down what your app on Render does with the same call, and which of the two you'd want in front of an agent at a client, and why.<br/><br/>
            Stretch challenge: Render can describe a service in a <code>render.yaml</code> file in the repository, called a Blueprint. Write one for the UnderCover app with the settings from Steps 2 and 3, leaving the secrets for the dashboard. Render's documentation: <a href="https://render.com/docs/infrastructure-as-code">Blueprints</a>.<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>